In [ ]:
# !pip install cyvcf2

In [ ]:
# CÉLULA DE TESTE DA BIBLIOTECA CYVCF2

from cyvcf2 import VCF
FILE_NAME = 'Y.vcf'

# Abre o arquivo .vcf (ou .vcf.gz)
vcf = VCF(FILE_NAME)

#Caso se queira visualizar somente o header
#print(vcf.raw_header)

#visualização das amostras participantes
# print("amostras: ", vcf.samples)

#Iterando sobre as variantes(linhas do Vcf)
for variant in vcf:

  #Impressão da linha completa
  print("Variante: ", variant)
  
  #Leitura de dados básicos
  cromossomo = variant.CHROM
  posicao = variant.POS
  id = variant.ID
  ref = variant.REF
  alt = variant.ALT
  quality = variant.QUAL

  #Acessando o campo INFO
  dp = variant.INFO.get('DP')
  ac = variant.INFO.get('AC')
  af = variant.INFO.get('AF')

  #Acessando os genótipos das amostras
  genotipos = variant.genotypes
  gt_types = variant.gt_types
  gt_ref_depths = variant.gt_ref_depths
  gt_alt_depths = variant.gt_alt_depths
  gt_phases = variant.gt_phases
  gt_quals = variant.gt_quals
  gt_bases = variant.gt_bases

  print(f'CHROM: {cromossomo}\nPOS: {posicao}\nID: {id}\nREF: {ref}\nALT: {alt}\nQUAL: {quality:.2f}\n')
  # print(genotipos)
  # print(gt_types)
  # print(gt_alt_depths)
  # print(gt_ref_depths)
  # print(gt_phases)
  # print(gt_quals)
  # print(gt_bases)

  #Forma de acessar informações especificas do campo FORMAT (para uma determinada amostra deve-se especificar o idx) 
  # (ex: os resultados de DP daquela amostra em todas as variantes)
  sample_idx = 1
  dp_array = variant.format('DP')
  dp = dp_array[sample_idx].tolist() if dp_array is not None else None
  print(dp)

  #Para imprimir apenas o primeiro
  break

vcf.close()


In [ ]:
from cyvcf2 import VCF
import pandas as pd
import os
from enum import Enum


FILE_NAME = 'Y.vcf'
FILE_PATH = f'/home/marcela/IC/IC/files/{FILE_NAME}'

VCF_FILE = VCF(FILE_PATH)

class BASIC_COLS(Enum):
    CHROM = 0
    POS = 1
    ID = 2
    REF = 3
    ALT = 4
    QUAL = 5
    FILTER = 6
    INFO = 7  
    FORMAT = 8


def vcf_to_df_raw(vcf_path, vcf_file):
    cmd = "zgrep '^#' " + vcf_path + "|tail -n 1"
    cols = os.popen(cmd).read().strip('#').strip('\n').split('\t')
    
    data = []
    
    for variant in vcf_file:
        raw_line = str(variant).rstrip('\n').split('\t')
        data.append(raw_line)

    df = pd.DataFrame(data, columns=cols)
    return df

def vcf_to_df_filtered(vcf_file):
    #Define as colunas desejadas e define subcolunas para pegar apenas parte de INFO
    cols_tuples = [
        ('#CHROM', ''),
        ('POS', ''),
        ('REF', ''),
        ('ALT', ''),
        ('INFO', 'AC'),  
        ('INFO', 'AF'),  
        ('INFO', 'DP'),  
        ('FORMAT', '')
    ]

    #Acrescenta as colunas de amostras
    for sample in vcf_file.samples:
        cols_tuples.append((sample, ''))

    multi_cols = pd.MultiIndex.from_tuples(cols_tuples)

    data = []
    for variant in vcf_file:
        raw_line = str(variant).strip('\n').split('\t')

        fltrd_line = [
        	raw_line[BASIC_COLS.CHROM.value],
            raw_line[BASIC_COLS.POS.value],
            raw_line[BASIC_COLS.REF.value],
            raw_line[BASIC_COLS.ALT.value],
            variant.INFO.get('AC'),
            variant.INFO.get('AF'),
            variant.INFO.get('DP'),
            raw_line[BASIC_COLS.FORMAT.value] #PROVISORIO! depois filtar o format tbm (pegar só os 4 prieiros)
        ]

        fltrd_line.extend(raw_line[BASIC_COLS.FORMAT.value+1:])
        data.append(fltrd_line)
    
    df = pd.DataFrame(data, columns=multi_cols)
    return df
    

# df_raw = vcf_to_df_raw(FILE_PATH, VCF_FILE)
# df_raw

df_flitered = vcf_to_df_filtered(VCF_FILE)
VCF_FILE.close()

df_flitered



# ajustar o format para pegar só (GT:AD:AF:DP) e os dados correspondentes de cada amostra
# terei problemas com arquivos maioes usando essa implementação??
# filtragem por genotipo e afins, entender e ver se direto pelo cyvcf2 é melhor
# 